# Step 7: Agentic LLM Underwriting Report (LangGraph + Ollama/Qwen)
## AI-Assisted Health Risk Prediction for Life Insurance Underwriting

The final stage of the pipeline:
```
Model prediction  ->  SHAP explanation  ->  LLM agent  ->  underwriting report
```

A **LangGraph ReAct agent** (running a local **Qwen** model via **Ollama**)
reasons over each applicant's prediction + SHAP contributions and writes a
human-readable underwriting summary.

**The agent has 3 tools it chooses to call:**
1. `lookup_clinical_ranges` : interpret a vital value against published reference ranges
2. `get_risk_category` : map the predicted probability to an underwriting decision
3. `explain_shap_contribution` : turn a SHAP value into a plain-English risk statement

**Important:** the LLM only **explains** the model's output. It does NOT make the
mortality prediction itself, that comes from XGBoost (Step 5).

## 7.1 Prerequisites

**Before running this notebook:**
1. Install Ollama: https://ollama.com/download/windows
2. Pull the model (in a terminal): `ollama pull qwen2.5:7b`
3. Ensure the Ollama service is running (it starts automatically after install)

Packages (`langchain`, `langgraph`, `langchain-ollama`).

In [1]:
import json
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent

MODEL_NAME = 'qwen2.5:7b'   # change to qwen2.5:3b if you pulled the smaller one
print('Imports OK')

Imports OK


In [2]:
# Quick connectivity check to the local Ollama server
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print('Ollama is running. Installed models:', models)
    if not any(MODEL_NAME.split(':')[0] in m for m in models):
        print(f'WARNING: {MODEL_NAME} not found. Run:  ollama pull {MODEL_NAME}')
except Exception as e:
    print('Could not reach Ollama at localhost:11434.')
    print('Make sure Ollama is installed and running, then re-run this cell.')
    print('Error:', e)

Ollama is running. Installed models: ['qwen2.5:7b']


## 7.2 Load SHAP Explanations & Clinical Lookups

We load the per-applicant SHAP explanations saved in Step 6 and the clinical
reference/risk lookup JSON that the tools will query.

In [3]:
def find_dir(name):
    for base in [Path('.'), Path('..'), Path('..') / '..']:
        candidate = base / name
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not locate {name}.')

DATA_DIR = find_dir('data')
RESULTS_DIR = DATA_DIR / 'results'
CONFIG_DIR = find_dir('config')
REPORTS_DIR = DATA_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# SHAP explanations from Step 6
with open(RESULTS_DIR / 'shap_explanations.json') as f:
    shap_explanations = json.load(f)

# Clinical lookups
with open(CONFIG_DIR / 'clinical_lookups.json') as f:
    LOOKUPS = json.load(f)

CLINICAL_RANGES = LOOKUPS['clinical_ranges']
RISK_CATEGORIES = LOOKUPS['risk_categories']

print(f'Loaded {len(shap_explanations)} applicant explanations')
print(f'Clinical ranges for {len(CLINICAL_RANGES)} features')
print(f'Risk category bands: {len(RISK_CATEGORIES["bands"])}')

Loaded 8 applicant explanations
Clinical ranges for 12 features
Risk category bands: 5


### De-standardize feature values back to original clinical units

Step 3 applied StandardScaler (z-score) to continuous features, so the
`feature_value` in SHAP explanations is in scaled units (e.g., AGE=1.69 means
1.69 stds above mean). Tool 1 (`lookup_clinical_ranges`) needs real-world
values (e.g., age=80) to classify into clinical bands. We load the scaler's
mean/std and un-transform each feature value.

In [5]:
import joblib
import numpy as np

# Load the preprocessing metadata and the fitted preprocessor from Step 3
MODEL_DIR = DATA_DIR / 'model_ready'
with open(MODEL_DIR / 'preprocessing_metadata.json') as f:
    prep_meta = json.load(f)

preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')

# Extract mean and std from the numeric pipeline's StandardScaler
# The preprocessor is a ColumnTransformer; 'num' is the numeric sub-pipeline
num_pipeline = preprocessor.named_transformers_['num']
scaler = num_pipeline.named_steps['scale']
continuous_features = prep_meta['continuous_features']

# Build lookup: feature_name -> (mean, std)
scaler_params = {
    feat: (float(scaler.mean_[i]), float(scaler.scale_[i]))
    for i, feat in enumerate(continuous_features)
}
print(f'Loaded scaler params for {len(scaler_params)} continuous features.')
print('Example: AGE mean={:.1f}, std={:.1f}'.format(*scaler_params.get('AGE', (0,1))))

# De-standardize feature_value in each SHAP explanation record
for rec in shap_explanations:
    for c in rec['top_contributions']:
        feat_name = c['feature']
        # Only de-standardize continuous features that went through the scaler
        # (prefixed 'num__' in encoded names but raw names in SHAP since we stripped prefix)
        if feat_name in scaler_params:
            mean, std = scaler_params[feat_name]
            original_val = c['feature_value'] * std + mean
            c['feature_value_original'] = round(float(original_val), 2)
        else:
            # categorical one-hot (0/1) or features not scaled — keep as-is
            c['feature_value_original'] = c['feature_value']

# Show a sample to verify
example_contribs = shap_explanations[0]['top_contributions'][:4]
for c in example_contribs:
    print(f"  {c['feature']:20s} scaled={c['feature_value']:.4f} -> original={c['feature_value_original']}")

Loaded scaler params for 26 continuous features.
Example: AGE mean=48.2, std=18.8
  AGE                  scaled=1.6905 -> original=80.0
  MCQ160B_1.0          scaled=1.0000 -> original=1.0
  LBXSATSI             scaled=-1.0113 -> original=2.71
  LBXSCR               scaled=3.4537 -> original=1.1


## 7.3 Define the Agent's Tools

Each tool is a plain Python function decorated with `@tool` so LangGraph exposes
it to the LLM. The values come from the clinical lookup JSON (not hardcoded), so
the medical thresholds are auditable and easy to update. Note that the model
features were standardized in Step 3, so tools accept the ORIGINAL-scale value
when interpreting clinically (the LLM is told to reason about direction, and the
SHAP value already encodes impact).

In [6]:
@tool
def lookup_clinical_ranges(feature_code: str, value: float) -> str:
    """Interpret a clinical lab or vital-sign value against published reference
    ranges. feature_code is an NHANES code like 'LBXGH' (HbA1c), 'BP_SYS_MEAN'
    (systolic BP), 'BMXBMI' (BMI). value is the measured value in the feature's
    natural units. Returns the matching clinical band (e.g. 'diabetic') and the
    full reference ranges. Use this to explain what an applicant's measurement means."""
    info = CLINICAL_RANGES.get(feature_code)
    if not info:
        return f'No clinical reference range available for {feature_code}.'
    band_label = 'unknown'
    for b in info['bands']:
        lo = b['min'] if b['min'] is not None else float('-inf')
        hi = b['max'] if b['max'] is not None else float('inf')
        if lo <= value < hi:
            band_label = b['label']
            break
    ranges_txt = '; '.join(
        f"{b['label']}: {b['min']}-{b['max'] if b['max'] is not None else '+'}"
        for b in info['bands'])
    return (f"{info['name']} = {value} {info['unit']} -> category: '{band_label}'. "
            f"Reference ranges [{ranges_txt}]. Source: {info['source']}.")

In [ ]:
@tool
def get_risk_category(predicted_risk: float) -> str:
    """Map a predicted mortality probability (0..1) to a simplified life-insurance
    underwriting decision category and recommended action. Use this once to state
    the overall underwriting recommendation for the applicant."""
    for b in RISK_CATEGORIES['bands']:
        if b['min'] <= predicted_risk < b['max']:
            return (f"Predicted risk {predicted_risk:.1%} -> category '{b['category']}'. "
                    f"Recommended action: {b['action']}. Rating: {b['table_rating']}.")
    return f'Predicted risk {predicted_risk:.1%} could not be categorized.'

In [8]:
@tool
def explain_shap_contribution(feature_code: str, shap_value: float, direction: str) -> str:
    """Turn a single SHAP contribution into a plain-English risk statement.
    feature_code is the NHANES code, shap_value is its SHAP value (log-odds impact),
    direction is 'increases_risk' or 'decreases_risk'. Use this to describe how
    each important factor pushed the applicant's risk up or down."""
    info = CLINICAL_RANGES.get(feature_code)
    friendly = info['name'] if info else feature_code
    verb = 'increased' if direction == 'increases_risk' else 'decreased'
    strength = 'strongly' if abs(shap_value) > 1.0 else ('moderately' if abs(shap_value) > 0.3 else 'slightly')
    return (f"{friendly} {strength} {verb} the applicant's mortality risk "
            f"(SHAP impact = {shap_value:+.2f} in log-odds).")

In [9]:
# Quick sanity test of the tools (plain function calls, no LLM yet)
print(lookup_clinical_ranges.invoke({'feature_code': 'LBXGH', 'value': 6.8}))
print()
print(get_risk_category.invoke({'predicted_risk': 0.45}))
print()
print(explain_shap_contribution.invoke({'feature_code': 'AGE', 'shap_value': 2.23, 'direction': 'increases_risk'}))

Hemoglobin A1c (HbA1c) = 6.8 % -> category: 'diabetic'. Reference ranges [normal: 0-5.7; prediabetic: 5.7-6.5; diabetic: 6.5-+]. Source: ADA 2023 Standards of Care.

Predicted risk 45.0% -> category 'High Risk'. Recommended action: Approve only with significant loading or refer to senior underwriter. Rating: Table E-H.

Age strongly increased the applicant's mortality risk (SHAP impact = +2.23 in log-odds).


## 7.4 Build the LangGraph ReAct Agent

`create_react_agent` wires the Qwen chat model to our three tools. The agent
follows the ReAct loop: **reason -> pick a tool -> observe result -> repeat ->
final answer**. The system prompt constrains it to underwriting explanation only.

In [10]:
llm = ChatOllama(model=MODEL_NAME, temperature=0.2)

SYSTEM_PROMPT = '''You are an AI underwriting assistant for a life insurance company.
You receive a mortality-risk prediction (already produced by a machine-learning
model) together with SHAP explanations of the factors that drove it.

Your job is to write a clear, professional underwriting summary. You MUST:
- Use get_risk_category once to state the overall underwriting decision.
- Use explain_shap_contribution for the most important factors.
- Use lookup_clinical_ranges to interpret specific clinical values when relevant.
- Never invent numbers. Only use values provided or returned by tools.
- NEVER change or re-estimate the mortality prediction yourself; you only explain it.

Write the final report in this structure:
1. Underwriting Decision (category + action)
2. Key Risk Factors (bulleted, most important first)
3. Protective Factors (if any)
4. Summary statement (2-3 sentences).'''

agent = create_react_agent(llm, tools=[lookup_clinical_ranges,
                                       get_risk_category,
                                       explain_shap_contribution])
print('Agent built with model:', MODEL_NAME)

Agent built with model: qwen2.5:7b


<TEMP>\ipykernel_28032\677801952.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[lookup_clinical_ranges,


## 7.5 Generate a Report for One Applicant

We format the applicant's SHAP record into a prompt and let the agent reason
and call tools to produce the underwriting summary.

In [11]:
def build_applicant_prompt(rec):
    lines = [f"Applicant ID: {rec['applicant_id']}",
             f"Predicted mortality risk: {rec['predicted_risk']:.4f}",
             '',
             'Top contributing factors (from SHAP):']
    for c in rec['top_contributions']:
        # Use original-scale value for clinical context (de-standardized in 7.2)
        orig_val = c.get('feature_value_original', c['feature_value'])
        lines.append(f"  - {c['feature']}: SHAP={c['shap_value']:+.3f}, "
                     f"value={orig_val}, {c['direction']}")
    lines.append('')
    lines.append('Write the underwriting summary. Use your tools to interpret the '
                 'risk category and the key factors. When calling '
                 'lookup_clinical_ranges, use the value shown above (original scale).')
    return '\n'.join(lines)

example = shap_explanations[0]
prompt = build_applicant_prompt(example)
print(prompt)

Applicant ID: 118
Predicted mortality risk: 0.9939

Top contributing factors (from SHAP):
  - AGE: SHAP=+2.232, value=80.0, increases_risk
  - MCQ160B_1.0: SHAP=+0.418, value=1.0, increases_risk
  - LBXSATSI: SHAP=+0.287, value=2.71, increases_risk
  - LBXSCR: SHAP=+0.278, value=1.1, increases_risk
  - MCQ160F_1.0: SHAP=+0.229, value=1.0, increases_risk
  - LBDHDD: SHAP=+0.211, value=3.3, increases_risk
  - RACE_ETHNICITY_3.0: SHAP=+0.206, value=1.0, increases_risk
  - MCQ220_1.0: SHAP=+0.183, value=1.0, increases_risk

Write the underwriting summary. Use your tools to interpret the risk category and the key factors. When calling lookup_clinical_ranges, use the value shown above (original scale).


In [12]:
# Invoke the agent (this calls the local Qwen model; may take ~30-90s on CPU)
result = agent.invoke({'messages': [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=prompt),
]})

final_report = result['messages'][-1].content
print('=' * 60)
print(f'UNDERWRITING REPORT — Applicant {example["applicant_id"]}')
print(f'(Actual outcome: {"Deceased" if example["actual_outcome"]==1 else "Alive"})')
print('=' * 60)
print(final_report)

UNDERWRITING REPORT — Applicant 118
(Actual outcome: Deceased)
1. Underwriting Decision (category + action)
   - Category: Decline
   - Action: Decline or postpone; refer for medical review

2. Key Risk Factors:
   - Age 80 years: This significant increase in age substantially elevates the mortality risk.
   - MCQ160B_1.0 and MCQ160F_1.0 (likely related to smoking): Both factors contribute positively, indicating a higher risk profile.
   - Low HDL Cholesterol (3.3 mg/dL): This is categorized as 'low (higher risk)', which further increases the overall mortality risk.
   - Elevated Serum Creatinine (1.1 mg/dL): While within the normal range, it still indicates potential renal impairment, contributing to higher risk.

3. Protective Factors:
   - None identified in this case.

4. Summary statement:
The applicant's high predicted mortality risk of 99.4% is driven primarily by advanced age and multiple risk factors including low HDL cholesterol and elevated serum creatinine. These factors, a

In [13]:
# Inspect the agent's tool-calling trace (shows the 'agentic' reasoning)
print('Agent message trace:')
for m in result['messages']:
    role = type(m).__name__.replace('Message', '')
    if getattr(m, 'tool_calls', None):
        for tc in m.tool_calls:
            print(f'  [{role}] TOOL CALL: {tc["name"]}({tc["args"]})')
    elif m.content:
        preview = m.content[:120].replace(chr(10), ' ')
        print(f'  [{role}] {preview}...')

Agent message trace:
  [System] You are an AI underwriting assistant for a life insurance company. You receive a mortality-risk prediction (already prod...
  [Human] Applicant ID: 118 Predicted mortality risk: 0.9939  Top contributing factors (from SHAP):   - AGE: SHAP=+2.232, value=80...
  [AI] TOOL CALL: get_risk_category({'predicted_risk': 0.9939})
  [AI] TOOL CALL: lookup_clinical_ranges({'feature_code': 'LBXSATSI', 'value': 2.71})
  [AI] TOOL CALL: lookup_clinical_ranges({'feature_code': 'LBXSCR', 'value': 1.1})
  [AI] TOOL CALL: lookup_clinical_ranges({'feature_code': 'LBDHDD', 'value': 3.3})
  [AI] TOOL CALL: get_risk_category({'predicted_risk': 0.9939})
  [Tool] Predicted risk 99.4% -> category 'Decline'. Recommended action: Decline or postpone; refer for medical review. Rating: D...
  [Tool] No clinical reference range available for LBXSATSI....
  [Tool] Serum Creatinine = 1.1 mg/dL -> category: 'normal'. Reference ranges [normal: 0-1.3; mildly elevated: 1.3-2.0; elevated ...


## 7.6 Generate Reports for All Sampled Applicants

Run the agent across the sampled applicants (high-risk, low-risk, mid-risk)
and save every report to disk for the ACM report appendix.

In [14]:
reports = []
for rec in shap_explanations:
    prompt = build_applicant_prompt(rec)
    res = agent.invoke({'messages': [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt),
    ]})
    report_text = res['messages'][-1].content
    reports.append({
        'applicant_id': rec['applicant_id'],
        'predicted_risk': rec['predicted_risk'],
        'actual_outcome': rec['actual_outcome'],
        'report': report_text,
    })
    print(f"Done applicant {rec['applicant_id']} (risk {rec['predicted_risk']:.2f})")

print(f'\nGenerated {len(reports)} reports.')

Done applicant 118 (risk 0.99)
Done applicant 12662 (risk 0.00)
Done applicant 6339 (risk 0.11)
Done applicant 8673 (risk 0.11)
Done applicant 10891 (risk 0.11)
Done applicant 17300 (risk 0.11)
Done applicant 1176 (risk 0.11)
Done applicant 16988 (risk 0.11)

Generated 8 reports.


In [15]:
# Save all reports (JSON + readable markdown)
with open(RESULTS_DIR / 'underwriting_reports.json', 'w') as f:
    json.dump(reports, f, indent=2)

md_lines = ['# AI-Generated Underwriting Reports\n',
            f'Model: XGBoost mortality prediction + SHAP + {MODEL_NAME} (LangGraph agent)\n']
for r in reports:
    outcome = 'Deceased' if r['actual_outcome'] == 1 else 'Alive'
    md_lines.append(f"\n---\n\n## Applicant {r['applicant_id']} "
                    f"(predicted risk {r['predicted_risk']:.1%}, actual: {outcome})\n")
    md_lines.append(r['report'])

report_md = REPORTS_DIR / 'underwriting_reports.md'
report_md.write_text('\n'.join(md_lines), encoding='utf-8')
print(f'Saved:\n  - {RESULTS_DIR / "underwriting_reports.json"}\n  - {report_md}')

Saved:
  - <PROJECT_ROOT>\data\results\underwriting_reports.json
  - <PROJECT_ROOT>\data\reports\underwriting_reports.md


## Wrap-up & Report Notes

**What this demonstrates (for the ACM report / presentation):**
- A genuine **agentic** step: the LLM autonomously decides which tools to call and
  in what order (visible in the tool-call trace), rather than following a fixed script.
- Clean **separation of concerns**: XGBoost predicts, SHAP explains numerically,
  the LLM translates into human-readable underwriting language. The LLM never predicts.
- **Grounded** clinical interpretation: tools pull authoritative reference ranges
  from a JSON config, reducing hallucination.

**Limitations to note:** local model quality varies; risk-category thresholds are
illustrative; reports should be reviewed by a human underwriter (human-in-the-loop).

This completes the modeling pipeline (Steps 1-7). Remaining: report writing (Step 9)
and presentation (Step 10).